# 01 — Download + parse NHANES III lab.dat

Pull the single ASCII file from CDC, parse just the variables the
analysis needs (SEQN, age, sex, phase, MEC weight, pseudo-PSU,
pseudo-stratum, LPP, LPPSI), and save a tidy parquet for the next
notebook.

The fixed-width column positions come from
[lab.sas](https://wwwn.cdc.gov/nchs/data/nhanes3/1a/lab.sas), the
official SAS read-in script. LPP is at columns 1643–1645 (1-based
→ 0-based slice `1642:1645`); LPPSI at 1646–1649 in g/L · 100
(format `7.2`).

**Heads-up.** The codebook (`lab2-acc.pdf`) lists the topic code
`LP` for lipoprotein(a) but the *variable* lives in the original
Release 1A `lab.dat`, not the 1998 supplement `lab2.dat`. The
variable was only populated for Phase II respondents (1991–94)
even though the file covers 1988–94.

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(os.path.abspath(os.path.join('..', 'data')))
RAW_DIR = DATA_DIR / 'raw' / 'nhanes' / 'nhanes3'
DERIVED_DIR = DATA_DIR / 'derived'
RAW_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

URL = 'https://wwwn.cdc.gov/nchs/data/nhanes3/1a/lab.dat'
LAB_PATH = RAW_DIR / 'lab.dat'

In [2]:
if not LAB_PATH.exists():
    print(f'Downloading {URL} → {LAB_PATH} (~56 MB)...')
    req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as r:
        LAB_PATH.write_bytes(r.read())
    print(f'  saved {LAB_PATH.stat().st_size:,} bytes')
else:
    print(f'cached: {LAB_PATH} ({LAB_PATH.stat().st_size:,} bytes)')

cached: /home/abie/csu_mace_rct_sim/ai_assisted_us_health_data_analysis/data/raw/nhanes/nhanes3/lab.dat (57,945,628 bytes)


## Parse the columns we need

Per `lab.sas` lines 14–34 (demographics + survey design), 281–282
+ 917–918 (Lp(a)). Positions are 1-based and inclusive in the SAS
spec; pandas `read_fwf` takes 0-based half-open slices, hence the
`(start-1, stop)` pattern below.

In [3]:
# (sas_position, name, dtype-or-divisor, label)
FIELDS = [
    ((1, 5),     'SEQN',     'str',        'sample sequence number'),
    ((15, 15),   'HSSEX',    'cat',        'sex (1=Male, 2=Female)'),
    ((16, 17),   'HSAGEIR',  'int',        'age at interview (screener)'),
    ((18, 18),   'HSAGEU',   'cat',        'age unit (1=Months, 2=Years)'),
    ((40, 40),   'SDPPHASE', 'cat',        'phase (1=1988-91, 2=1991-94)'),
    ((41, 41),   'SDPPSU6',  'int',        'NHANES III pseudo-PSU'),
    ((42, 43),   'SDPSTRA6', 'int',        'NHANES III pseudo-stratum'),
    ((50, 58),   'WTPFQX6',  ('float', 100),    'final interview weight (impl. dec. 2)'),
    ((59, 67),   'WTPFEX6',  ('float', 100),    'final MEC-examined weight (impl. dec. 2)'),
    ((68, 76),   'WTPFHX6',  ('float', 100),    'final MEC + home weight (impl. dec. 2)'),
    ((1643, 1645), 'LPP',    'int',        'serum lipoprotein(a), mg/dL'),
    ((1646, 1649), 'LPPSI',  ('float', 100),    'serum lipoprotein(a), g/L (impl. dec. 2)'),
]

colspecs = [(s - 1, e) for (s, e), _, _, _ in FIELDS]
names = [n for _, n, _, _ in FIELDS]

df = pd.read_fwf(LAB_PATH, colspecs=colspecs, names=names, dtype=str)
print(f'parsed {len(df):,} rows')
df.head()

parsed 29,314 rows


,SEQN,HSSEX,HSAGEIR,HSAGEU,SDPPHASE,SDPPSU6,SDPSTRA6,WTPFQX6,WTPFEX6,WTPFHX6,LPP,LPPSI
0,00003,1,21,2,1,1,44,000001523,001737.58,001735.14,NaN,NaN
1,00004,2,32,2,1,1,43,001566.14,001730.23,001725.01,NaN,NaN
2,00007,2,03,2,1,1,43,001252.53,001241.64,001242.59,NaN,NaN
3,00009,2,48,2,1,2,43,018155.15,019521.36,019451.83,NaN,NaN
4,00010,1,35,2,1,1,06,022220.17,028080.57,027769.56,NaN,NaN


In [4]:
# Numeric coercions and missing-code handling.
#
# Per the NHANES III lab codebook conventions, Lp(a) values like
# 888 / 8888 indicate "could not be assayed / refused". 0 mg/dL is
# the detection limit so true zeros are rare-but-possible. We treat
# 888 as missing; everything else passes through.
df['LPP'] = pd.to_numeric(df['LPP'], errors='coerce')
df['LPPSI'] = pd.to_numeric(df['LPPSI'], errors='coerce') / 100.0
df.loc[df['LPP'] == 888, 'LPP'] = np.nan
df.loc[df['LPPSI'].isin([8.88]), 'LPPSI'] = np.nan

for col in ['HSAGEIR', 'SDPPSU6', 'SDPSTRA6']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in ['WTPFQX6', 'WTPFEX6', 'WTPFHX6']:
    df[col] = pd.to_numeric(df[col], errors='coerce') / 100.0

df['HSSEX'] = df['HSSEX'].map({'1': 'Male', '2': 'Female'})
df['HSAGEU'] = df['HSAGEU'].map({'1': 'Months', '2': 'Years'})
df['SDPPHASE'] = df['SDPPHASE'].map({
    '1': 'Phase 1 (1988-91)',
    '2': 'Phase 2 (1991-94)',
})

# age in years (drop infants under 1 year for the Lp(a) work)
df['age_years'] = np.where(df['HSAGEU'] == 'Years', df['HSAGEIR'], np.nan)

df.head()

,SEQN,HSSEX,HSAGEIR,HSAGEU,SDPPHASE,SDPPSU6,SDPSTRA6,WTPFQX6,WTPFEX6,WTPFHX6,LPP,LPPSI,age_years
0,00003,Male,21,Years,Phase 1 (1988-91),1,44,15.2300,17.3758,17.3514,NaN,NaN,21.0
1,00004,Female,32,Years,Phase 1 (1988-91),1,43,15.6614,17.3023,17.2501,NaN,NaN,32.0
2,00007,Female,3,Years,Phase 1 (1988-91),1,43,12.5253,12.4164,12.4259,NaN,NaN,3.0
3,00009,Female,48,Years,Phase 1 (1988-91),2,43,181.5515,195.2136,194.5183,NaN,NaN,48.0
4,00010,Male,35,Years,Phase 1 (1988-91),1,6,222.2017,280.8057,277.6956,NaN,NaN,35.0


In [5]:
print('Total NHANES III lab.dat rows:', f'{len(df):,}')
print('Phase coverage of LPP measurement:')
print(df.groupby('SDPPHASE', dropna=False)['LPP'].agg(['count', 'mean', 'median', 'std']).round(1))
print()
print('Sex distribution (LPP-measured Phase 2 adults 20+):')
print(
    df[(df['LPP'].notna()) & (df['age_years'] >= 20)]
    .groupby('HSSEX')['LPP']
    .agg(['count', 'mean', 'median', 'std'])
    .round(1)
)

Total NHANES III lab.dat rows: 29,314
Phase coverage of LPP measurement:
                   count  mean  median   std
SDPPHASE                                    
Phase 1 (1988-91)      0   NaN     NaN   NaN
Phase 2 (1991-94)  12018  26.2    18.0  28.5

Sex distribution (LPP-measured Phase 2 adults 20+):
        count  mean  median   std
HSSEX                            
Female   4641  27.8    19.0  30.0
Male     3576  25.0    16.0  27.9


In [6]:
out = DERIVED_DIR / 'nhanes3_lpa.parquet'
df.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size:,} bytes)')
df.dtypes

wrote /home/abie/csu_mace_rct_sim/ai_assisted_us_health_data_analysis/data/derived/nhanes3_lpa.parquet (671,784 bytes)


SEQN          object
HSSEX         object
HSAGEIR        int64
HSAGEU        object
SDPPHASE      object
SDPPSU6        int64
SDPSTRA6       int64
WTPFQX6      float64
WTPFEX6      float64
WTPFHX6      float64
LPP          float64
LPPSI        float64
age_years    float64
dtype: object